# A08 - Perceptrón Multicapa

|                |                  |
:----------------|------------------|
| **Nombre**     | Omar Mendoza     |
| **Fecha**      | 23/10/2026       |
| **Expediente** | 757806           |

Escoger hiperparametros es un problema de optimizacion, se prueban varias combinaciones y te quedas con la que saque mejor calificacion. Como en el ejemplo del pizarron con la SVM, 3 valores de gamma y 3 de C dan 9 modelos y gana el que saca mas

Aqui con el dataset del corazon se busca cuantas neuronas ponerle a un perceptron de 3 capas ocultas:

- Capa 1: de 5 a 15 neuronas
- Capa 2: de 10 a 20 neuronas
- Capa 3: de 5 a 10 neuronas

Son 11 x 11 x 6 = 726 redes. Cada una se califica con el AUC promedio de un K-Fold de 5 y se queda la mejor

$$\max_{c_1,\, c_2,\, c_3} \; \frac{1}{5} \sum_{k=1}^{5} AUC_k(c_1, c_2, c_3)$$

Al final la ganadora se prueba con datos de test que nunca vio

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

In [2]:
datos = pd.read_csv("Heart Prediction Quantum Dataset.csv")

print(datos.shape)
datos.head()

(500, 6)


,Age,Gender,BloodPressure,Cholesterol,HeartRate,HeartDisease
0,68,Hombre,105.00,191,107,1
1,58,Mujer,97.00,249,89,0
2,44,Mujer,93.00,190,82,1
3,72,Hombre,93.00,183,101,1
4,37,Mujer,145.00,166,103,1


Son 500 pacientes y 6 columnas, edad, genero, presion, colesterol, ritmo cardiaco y si tiene la enfermedad. Ya se ve que Gender trae espacios raros

In [3]:
print(datos.dtypes)
print()
print("Nulos:", datos.isna().sum().sum())
print("Duplicados:", datos.duplicated().sum())

Age               int64
Gender           object
BloodPressure    object
Cholesterol       int64
HeartRate         int64
HeartDisease      int64
dtype: object

Nulos: 0
Duplicados: 0


In [4]:
print(datos["HeartDisease"].value_counts())
print()
print(datos["Gender"].value_counts())

HeartDisease
1    300
0    200
Name: count, dtype: int64

Gender
Mujer        208
Hombre       149
Hombre        66
mujer         43
Hombre}       18
MUJER          7
Mujer{         5
MUjer          3
ombre          1
Name: count, dtype: int64


No hay nulos ni duplicados. Hay 300 enfermos y 200 sanos, no estan parejos y por eso conviene el AUC. BloodPressure viene como texto porque trae comas y Gender esta escrito de muchas formas

In [5]:
datos["BloodPressure"] = datos["BloodPressure"].str.replace(",", ".")
datos["BloodPressure"] = pd.to_numeric(datos["BloodPressure"])

genero = datos["Gender"].str.lower()
genero = genero.str.rstrip(" {}")
genero[genero == "ombre"] = "hombre"
datos["Gender"] = (genero == "hombre").astype(int)

print(datos["Gender"].value_counts())
datos.describe()

Gender
0    266
1    234
Name: count, dtype: int64


,Age,Gender,BloodPressure,Cholesterol,HeartRate,HeartDisease
count,500.000000,500.000000,500.000000,500.00000,500.000000,500.000000
mean,54.864000,0.468000,132.874000,221.50000,88.766000,0.600000
std,14.315004,0.499475,26.418516,43.86363,17.417289,0.490389
min,30.000000,0.000000,90.000000,150.00000,60.000000,0.000000
25%,43.000000,0.000000,111.000000,183.75000,73.000000,0.000000
50%,55.000000,0.000000,132.000000,221.00000,89.000000,1.000000
75%,66.250000,1.000000,155.000000,258.00000,104.000000,1.000000
max,79.000000,1.000000,179.000000,299.00000,119.000000,1.000000


Se cambia la coma por punto en la presion y Gender queda 1 hombre y 0 mujer, salieron 234 hombres y 266 mujeres asi que no se perdio ninguno. Las variables estan en escalas muy distintas, el colesterol va de 150 a 299 y la edad de 30 a 79, por eso hay que escalar

In [6]:
X = datos.drop(columns="HeartDisease")
y = datos["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(X_train.shape, X_test.shape)

(350, 5) (150, 5)


Quedan 350 pacientes para entrenar y 150 para probar. El K-Fold solo usa los 350 de train para escoger la red, los de test se guardan para el final

In [7]:
numerical_cols = ["Age", "BloodPressure", "Cholesterol", "HeartRate"]
boolean_cols = ["Gender"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("bool", "passthrough", boolean_cols),
    ]
)

pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("mlp", MLPClassifier(max_iter=500, random_state=42))])

Se escalan las numericas y Gender pasa tal cual porque ya es 0 y 1. Va todo en un pipeline para que el escalamiento se haga dentro de cada fold. Le puse max_iter de 500 para que tenga mas vueltas que las 200 de default

In [8]:
capas = []

for c1 in range(5, 16):
    for c2 in range(10, 21):
        for c3 in range(5, 11):
            capas.append((c1, c2, c3))

print(len(capas))
print(capas[0], capas[-1])

param_grid = {"mlp__hidden_layer_sizes": capas}

726
(5, 10, 5) (15, 20, 10)


Con los tres for salen las 726 redes, desde (5, 10, 5) hasta (15, 20, 10). El mlp__ es porque asi se llama el paso de la red en el pipeline

In [9]:
grid = GridSearchCV(pipeline, param_grid, scoring="roc_auc", cv=5)

%time grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)

CPU times: total: 19min 14s
Wall time: 19min 48s
{'mlp__hidden_layer_sizes': (5, 14, 7)}
0.8748067603828747


GridSearchCV entrena cada red 5 veces, son 3630 entrenamientos y por eso tardo casi 20 minutos. La mejor fue la de 5, 14 y 7 neuronas con 0.875 de AUC promedio. No es la mas grande, mas neuronas no siempre es mejor

In [10]:
modelo = grid.best_estimator_

proba = modelo.predict_proba(X_test)[:, 1]
y_pred = modelo.predict(X_test)

print("AUC de test:", roc_auc_score(y_test, proba))

AUC de test: 0.8098342335630471


Con test el AUC bajo a 0.810. Es normal que baje porque se escogio la mejor de 726 y parte de esa ventaja fue suerte, pero sigue estando bien, 8 de cada 10 veces le da mas probabilidad al enfermo que al sano

In [11]:
print(classification_report(y_test, y_pred, target_names=["Sin enfermedad", "Con enfermedad"]))

                precision    recall  f1-score   support

Sin enfermedad       0.69      0.64      0.67        59
Con enfermedad       0.78      0.81      0.80        91

      accuracy                           0.75       150
     macro avg       0.73      0.73      0.73       150
  weighted avg       0.74      0.75      0.74       150



De los 91 enfermos detecta el 81 por ciento y cuando dice enfermo acierta el 78. Con los sanos le va peor, reconoce el 64 por ciento. El accuracy es 0.75, acierta 112 de 150

In [12]:
# renglones lo real, columnas lo predicho

confusion_matrix(y_test, y_pred)

array([[38, 21],
       [17, 74]])

Atino 38 sanos y 74 enfermos. Se equivoco con 21 sanos que marco como enfermos y 17 enfermos que se le pasaron, esos 17 son los que mas importan porque les diria que estan bien